<a href="https://colab.research.google.com/github/MOSAB-ABAKAR/Notebook-AIagent/blob/main/NoteTaker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pretrained Model: AI Notetaker

In this notebook we will build our AI Notetaker from scratch.

## Goals for this notebook:
1. Learn how to build the individual components of our overall app
2. Learn how to build a basic chat logic
3. Learn how Gradio works

# Step 1: Install and Import Libraries

In [ ]:
!pip install -q torch transformers bitsandbytes huggingface_hub gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.5 MB/s eta 0:00:00


In [ ]:
import os
import torch
from google.colab import userdata
from huggingface_hub import login

OPENAI_KEY = userdata.get("OPENAI_API_KEY")
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

# Step 2: Download our Model from Huggingface

Watch out for pad_token_id=128001. This is particularly for the model `meta-llama/Llama-3.1-8B-Instruct`.

If you were to choose another model, you will have to get the correct eos_token_id for that model.

In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 56.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0


In [ ]:
from transformers import pipeline, BitsAndBytesConfig

chatbot = pipeline(
    model="meta-llama/Llama-3.1-8B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    model_kwargs={"quantization_config": BitsAndBytesConfig(load_in_4bit=True)},
    pad_token_id=128001
    )

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

# Step 3: Model Inference
## 3.1 Basic Prompting

In [ ]:
prompt = "What is a current existential crisis faced by humanity?"

messages = [
    {'role':'user','content':prompt}
]
messages

[{'role': 'user',
  'content': 'What is a current existential crisis faced by humanity?'}]

In [ ]:
response = chatbot(messages, max_new_tokens=50)
print(response[0]['generated_text'][-1]['content'])

## 3.2 Chat Logic from Scratch

In [ ]:
messages = []
user_input= ""

while user_input != 'bye':
  user_input = str(input("You: "))
  messages.append({'role':'user', 'content':user_input})
  ai_response = chatbot(messages, max_new_tokens=50)
  ai_response = ai_response[0]['generated_text'][-1]['content']
  print("AI: ", ai_response)
  messages.append({'role':'assistant','content':ai_response})

You: hello
AI:  Hello! How are you doing today? Is there something I can help you with or would you like to chat?
You: yes. can you explain the role of ai models
AI:  AI models, also known as artificial intelligence models, are a type of computer program that uses machine learning algorithms to analyze and learn from data. These models are designed to perform specific tasks, such as:

1. **Pattern recognition**: Identifying patterns in
You: bye
AI:  Goodbye! It was nice chatting with you. If you have any more questions or need further assistance, feel free to come back anytime. Have a great day!


**A deeper look at the chatbot message history**

Run the following cell and chat with the model to know how the chat history looks as you continue your conversation. Note that the AI is passed the **entire** conversation as context whenever an API call is made.

In [ ]:
from pprint import pprint
messages = []
user_input = ''

while user_input != 'bye':
  user_input = input('You: ')
  messages.append({'role':'user','content':user_input})
  print('\nWhat is being passed to the model?\n')
  pprint(messages)
  print('\n')
  assistant = chatbot(messages)[0]['generated_text'][-1]['content']
  print("AI: ", assistant)
  messages.append({'role':'assistant','content':assistant})

You: hi there

What is being passed to the model?

[{'content': 'hi there', 'role': 'user'}]


AI:  How are you today? Is there something I can help you with or would you like to chat?
You: what time in the morning is best for working out?

What is being passed to the model?

[{'content': 'hi there', 'role': 'user'},
 {'content': 'How are you today? Is there something I can help you with or '
             'would you like to chat?',
  'role': 'assistant'},
 {'content': 'what time in the morning is best for working out?',
  'role': 'user'}]


AI:  Research suggests that the best time for a workout is early in the morning, typically between 5-
You: please continue your thought

What is being passed to the model?

[{'content': 'hi there', 'role': 'user'},
 {'content': 'How are you today? Is there something I can help you with or '
             'would you like to chat?',
  'role': 'assistant'},
 {'content': 'what time in the morning is best for working out?',
  'role': 'user'},
 {'content':

Wrapping our chat functionality into a function acceptible for Gradio

In [ ]:
def chat(messages, history):
  history.append({'role':'user','content':messages})
  ai_response = chatbot(history, max_new_tokens=50)[0]['generated_text'][-1]['content']
  return ai_response

# Step 4: A look at Gradio

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn = chat
)

demo.launch(debug=True)

# Step 5: Audio Transcriber

In [ ]:
from openai import OpenAI
from pathlib import Path
from google.colab import userdata

try:
    OPENAI_KEY = userdata.get("OPENAI_API_KEY")
except Exception as e:
    OPENAI_KEY = None
    print("Warning: Could not retrieve OPENAI_API_KEY from secrets.")

openai_client = OpenAI(api_key=OPENAI_KEY)

def transcribe_audio(audio):
  audio_file = Path(audio)
  transcriber = openai_client.audio.transcriptions.create(
      model="whisper-1",
      file=audio_file
  )

  transcription = transcriber.text
  return transcription

In [ ]:
def note_form_audio(audio=None):
  transcription = transcribe_audio(audio)

  SYSTEM_PROMPT_MESSAGE = "You are an expert notetaker, use the following transcription from my audio to generate "
  SYSTEM_PROMPT_MESSAGE += "notes based on it. Make sure to keep these brief using bullet points"
  prompt = SYSTEM_PROMPT_MESSAGE + "\nTranscription\n" + transcription
  messages = [
        {'role':'user','content':prompt}
    ]
  # The chat function defined earlier requires two arguments: messages and history
  notes = chat(messages, [])
  return notes, transcription

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn = chat
)

demo.launch(debug=True)

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
  with gr.Row():
    audio_input = gr.Audio(sources=["upload","microphone"], type="filepath", label="Audio Transcriber")

  with gr.Row():
    transcription = gr.Textbox(label="Transcription")
    notes = gr.Textbox(label="Notes")

  with gr.Row():
    submit_button = gr.Button("Submit")
    submit_button.click(note_form_audio, inputs=[audio_input], outputs=[notes, transcription])
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://652cb5791d77491355.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
